# System Router on Colab

Thin notebook: all logic lives in the repository. Use an A100 runtime for Qwen3.5-9B unquantized; on a T4 or L4 add `--set llm.quantization=4bit` (and `llm.dtype=float16` on a T4).

Runs, the collected dataset and checkpoints go to Google Drive so a disconnect loses nothing; the model is downloaded to the session disk each time. Stage 3 (offline training) needs no model at all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/system-router/runs'
MODEL = '/content/models/Qwen3.5-9B'
SET = f'--set paths.model={MODEL} --set paths.runs={RUNS} --set collect.out={RUNS}/dataset --set heads.checkpoint={RUNS}/offline/latest.pt --set paths.label_cache={RUNS}/label_cache'

In [ ]:
!git clone --recurse-submodules https://github.com/serciex/System-Router.git /content/System-Router
%cd /content/System-Router

In [ ]:
# Chrome for MiniWoB++ (Selenium downloads a matching chromedriver itself)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq update && apt-get -qq install -y ./google-chrome-stable_current_amd64.deb
!pip -q install -r requirements.txt

In [ ]:
# Stage 0: unit tests and adapter conformance (no model needed)
!python -m pytest -q tests
!python scripts/check_adapters.py --adapters dom a11y "degraded:dom"

In [ ]:
# Model (only needed for stages 1, 2, 4 and 5)
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3.5-9B', local_dir=MODEL)

In [ ]:
# Stage 1: rules instead of the world model, System 1 vs System 2 alone
!python scripts/run_stage1.py --episodes 20 --route s1 {SET}
!python scripts/run_stage1.py --episodes 20 --route s2 {SET}

In [ ]:
# Stage 2: collect once (both systems answer every step); rerun with --seed-offset to extend
!python scripts/collect.py --episodes 500 {SET} --set 'env.adapter_pool=[[dom],[a11y],["degraded:dom"]]'
!python scripts/collect.py --episodes 100 --heldout {SET} --set 'env.heldout_adapters=[["degraded:a11y"]]'

In [ ]:
# Stage 3: offline world model + supervised heads, and the plain-classifier baseline (no LLM)
!python scripts/train_offline.py {SET}
!python scripts/train_offline.py --baseline {SET}

In [ ]:
# Stage 4: online with the trained heads (thresholds from tuning.json)
!python scripts/run_stage1.py --episodes 20 --policy heads {SET}
!python scripts/run_stage1.py --episodes 20 --policy heads --heldout {SET} --set 'env.heldout_adapters=[["degraded:a11y"]]'

In [ ]:
# Stage 5 (later): RL fine-tuning from the offline model
!python scripts/train_wm.py --init-from {RUNS}/offline/latest.pt {SET}